### A5.1.11. Build RAII File Descriptor Wrapper

**Problem:**

Build a wrapper class that acquires a file descriptor on creation and guarantees it is closed on destruction, following the RAII (Resource Acquisition Is Initialization) pattern via Python's context manager protocol.

**Explanation:**

RAII ties resource lifetime to object lifetime. In Python, the context manager protocol (`__enter__`/`__exit__`) provides the same guarantee: the resource is acquired when entering the `with` block and released when leaving it, even if an exception occurs.

How to build it:

1. In `__init__`, accept a file path and mode. Open the file using `os.open` to get a raw file descriptor.
2. In `__enter__`, return `self` so the caller can use the wrapper.
3. Provide `read` and `write` methods that use `os.read` and `os.write` on the stored file descriptor.
4. In `__exit__`, call `os.close` on the file descriptor to release it.
5. The file descriptor is always closed when leaving the `with` block.

**Example:**

Open a file, write bytes, read them back, and the descriptor is closed automatically when the `with` block ends.

In [ ]:
import os
import tempfile


class FileDescriptorWrapper:
    def __init__(self, path, flags):
        self.fd = os.open(path, flags, 0o644)

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        os.close(self.fd)
        return False

    def write(self, data):
        return os.write(self.fd, data)

    def read(self, size):
        return os.read(self.fd, size)


tmp_path = os.path.join(tempfile.gettempdir(), "raii_test.bin")

with FileDescriptorWrapper(tmp_path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC) as writer:
    writer.write(b"RAII in Python")
    print(f"Wrote to fd={writer.fd}")

with FileDescriptorWrapper(tmp_path, os.O_RDONLY) as reader:
    content = reader.read(1024)
    print(f"Read: {content}")

os.unlink(tmp_path)

**References:**

📘 Stroustrup, B. (2013). *The C++ Programming Language* — RAII

📘 Python Documentation — [Context Manager Types](https://docs.python.org/3/reference/datamodel.html#context-managers)

---

[⬅️ Previous: Build Polymorphic Serializer Interface](./10_build_polymorphic_serializer_interface.ipynb) | [Next: Build Reference Counted Handle Class ➡️](./12_build_reference_counted_handle_class.ipynb)